# Workshop: real-gold 4D-STEM — browse, BF, DF, DPC, and 3 direct-ptycho kernels (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/bobleesj/a05a90185c6cddbb331342cae6d7e9c1/berk_workshop_v1.ipynb)

ONE notebook. Real gold from Hugging Face → load → browse → bright field → dark
field → DPC (via `CenterOfMassOriginModel`) → three single-shot phase-retrieval
kernels (parallax, SSB, ICOM) → side-by-side comparison.

Everything on torch on the Colab T4. Two installs only — `quantem.widget`
(TestPyPI prerelease) + `quantem` (`berk-workshop` branch). No `quantem.live`.

**Workshop punchline:** phase retrieval (parallax, SSB) recovers atomic-lattice
contrast that BF/DF physically cannot, at the same dose.

In [38]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [39]:
import quantem as em
import quantem.widget
import torch

# cuDNN grid_sample bug at these detector dims; disable for DirectPtycho path.
torch.backends.cudnn.enabled = False

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__, "(cuDNN disabled)")
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")

quantem         0.1.8
quantem.widget  0.0.1
torch           2.10.0+cu130 (cuDNN disabled)
cuda available: True NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [40]:
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.ascontiguousarray(np.load(os.path.join(asset, "data.npy")).astype(np.float32))
meta = json.load(open(os.path.join(asset, "meta.json")))

# numpy-backed for upstream CoM + DirectPtychography (they read dataset.array)
dset = em.core.datastructures.Dataset4dstem.from_array(
    data, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset: shape {dset.shape}, dtype {dset.array.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad, CL {meta['camera_length_mm']} mm")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

dataset: shape (512, 512, 24, 24), dtype float32
sampling [0.5, 0.5, 3.68, 3.68] ['A', 'A', 'mrad', 'mrad']
optics: 300 kV, probe 30 mrad, CL 91 mm


## Step 1 — Browse the 4D-STEM dataset interactively

Drag the scan cursor; CBED updates live. Real-time per-scan-position BF/DF.

In [41]:
# Torch-backed view for Show4DSTEM (GPU-fast cursor drag)
dset_torch = em.core.datastructures.Dataset4dstem.from_tensor(
    torch.from_numpy(data).to("cuda" if torch.cuda.is_available() else "cpu"),
    sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
w = quantem.widget.Show4DSTEM(dset_torch)
w.layout.width = "1000px"      # ipywidgets layout — best-effort; widget JS may override
w

  to cuda:0: 0.00s (0.6 GB)
  auto_detect_center: 0.00s
  virtual image + frame: 0.00s
Show4DSTEM: 512x512x24x24 cuda:0, 0.02s total


Show4DSTEM(shape=(512, 512, 24, 24), sampling=(0.5 A, 3.68 mrad), pos=(256, 256), title='gold_512_npy_bin8')

## Step 2 — Bright field + Dark field, side by side (independent contrast)

`Dataset4dstem.get_virtual_image(mode="circle"|"annular", ...)` makes the
aperture masks + per-scan-position sum. One `Show2D` widget with both panels;
`link_contrast=False` so each panel scales to its own min/max.

In [42]:
import math

H, W = dset.shape[-2:]
cy, cx = H / 2, W / 2                                    # hardcoded geometric center
BF_RADIUS_PX = 6.0
R_MAX = math.hypot(cy, cx)                                # corner-of-detector radius

bf_ds = dset.get_virtual_image(mode="circle",  geometry=((cy, cx), BF_RADIUS_PX), name="BF")
df_ds = dset.get_virtual_image(mode="annular", geometry=((cy, cx), (BF_RADIUS_PX, R_MAX)), name="DF")
print(f"BF range [{bf_ds.array.min():.1f}, {bf_ds.array.max():.1f}]")
print(f"DF range [{df_ds.array.min():.1f}, {df_ds.array.max():.1f}]")

quantem.widget.Show2D(
    [bf_ds.array, df_ds.array],
    labels=["Bright field", "Dark field"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
    link_contrast=False,
)

BF range [385193.0, 531066.0]


Show2D(512×512, cmap=gray)

## Step 3 — DPC via `CenterOfMassOriginModel`

Per-scan-position centroid (CoM) on the GPU. Two-cell call, two interactive panels.

In [44]:
from quantem.diffractive_imaging import CenterOfMassOriginModel

com_model = CenterOfMassOriginModel.from_dataset(dset, device="cuda" if torch.cuda.is_available() else "cpu")
com_model.calculate_origin()
com = com_model.origin_measured.view(*dset.shape[:2], 2).cpu().numpy()

quantem.widget.Show2D(
    [com[..., 0], com[..., 1]],
    labels=["CoM row (qx)", "CoM col (qy)"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="RdBu_r", link_contrast=False,
)

CoM row range [-0.2639, 0.2487] px
CoM col range [-0.1957, 0.3007] px
|CoM| max     0.3333 px


Show2D(3×512×512, idx=0, cmap=RdBu_r)

## Step 3b — Scan-detector rotation from CoM curl

`DirectPtychography` needs the rotation between the scan axes and the detector
axes. Find it by sweeping rotation angles and minimizing the curl of the
rotated CoM field. (Auto-estimated internally by `from_dataset4d` too — this
cell exposes the curve so you can see WHERE the minimum lands.)

In [ ]:
import matplotlib.pyplot as plt

# Fit origin (needed by estimate_detector_rotation) + run estimate.
com_model.fit_origin_background()
com_model.estimate_detector_rotation()
best_rotation_deg = com_model.detector_rotation_deg
best_rotation_rad = np.deg2rad(best_rotation_deg)
print(f"detector rotation = {best_rotation_deg:.2f}°  ({best_rotation_rad:.4f} rad)")

# Recompute curl-vs-angle for the plot.
angles_deg = torch.arange(-89, 90, 1, device=com_model.device, dtype=torch.float32)
com_t = com_model.origin_measured.view(*dset.shape[:2], 2)
com_n = com_t - com_t.mean(dim=(0, 1))
rot_rad = torch.deg2rad(angles_deg)[:, None, None]
rx = torch.cos(rot_rad) * com_n[None, ..., 0] - torch.sin(rot_rad) * com_n[None, ..., 1]
ry = torch.sin(rot_rad) * com_n[None, ..., 0] + torch.cos(rot_rad) * com_n[None, ..., 1]
grad_xy = torch.gradient(rx, dim=-1)[0]
grad_yx = torch.gradient(ry, dim=-2)[0]
curl = torch.mean(torch.abs(grad_yx - grad_xy), dim=(-2, -1)).cpu().numpy()
angles = angles_deg.cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(angles, curl, lw=1.5)
ax.axvline(best_rotation_deg, color="red", lw=1, label=f"{best_rotation_deg:.1f}°")
ax.set_xlabel("rotation angle (deg)")
ax.set_ylabel("mean |curl(CoM)|")
ax.set_title("Scan-detector rotation from CoM curl")
ax.legend()
plt.tight_layout(); plt.show()

## Step 4 — Phase retrieval: `DirectPtychography` + optuna aberration fit

Build `DirectPtychography` with the **rotation found in Step 4b** (`best_rotation_rad`),
then **fit C10, C12, phi12 from the data** with a 30-trial optuna loop (~5 s on T4).
The fitted aberration coefs go into the final reconstructions.

Why fit per-dataset: SSB silently returns zeros when the aberrations are wrong —
it needs the probe phase profile to deconvolve. Fitting on each dataset means the
workshop works on YOUR data, no prior calibration assumed.

In [45]:
from quantem.diffractive_imaging import DirectPtychography

direct = DirectPtychography.from_dataset4d(
    dset,
    energy=meta["voltage_kV"] * 1e3,                          # 300 kV -> 300000 eV
    semiangle_cutoff=meta["probe_semiangle_mrad"] * 1e-3,     # 30 mrad -> 0.030 rad
    rotation_angle=best_rotation_rad,                          # from Step 4b CoM curl
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)
print(f"DirectPtychography built with rotation = {np.rad2deg(best_rotation_rad):.2f}°")

DirectPtychography built


In [46]:
import optuna, time
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """Optuna objective: minimize SSB variance_loss over C10/C12/phi12."""
    c10   = trial.suggest_float("C10",   -300.0, 300.0)
    c12   = trial.suggest_float("C12",   -100.0, 100.0)
    phi12 = trial.suggest_float("phi12",    0.0, float(np.pi))
    direct.reconstruct(
        deconvolution_kernel="ssb",
        override_aberration_coefs={"C10": c10, "C12": c12, "phi12": phi12},
        parallax_flip_phase=False,
        verbose=False,
    )
    return float(direct.variance_loss())

t0 = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30, show_progress_bar=False)
print(f"optuna 30 trials: {time.time()-t0:.1f}s  best variance_loss: {study.best_value:.6f}")
print(f"fitted aberrations: {study.best_params}")

KERNELS = ["icom", "parallax", "ssb"]   # display order: ICOM (CoM-integrated), then parallax, then SSB
phases = {}
for k in KERNELS:
    t0 = time.time()
    direct.reconstruct(
        deconvolution_kernel=k,
        override_aberration_coefs=study.best_params,
        parallax_flip_phase=False,
        verbose=False,
    )
    phases[k] = direct.corrected_bf.detach().cpu().numpy()
    print(f"  {k:>9}: {time.time()-t0:.2f}s  range [{phases[k].min():.4f}, {phases[k].max():.4f}]  std={phases[k].std():.4f}")

   parallax: 0.29s  range [-120.609, 62.951]
        ssb: 0.27s  range [-0.274, 0.472]
       icom: 0.26s  range [-567.473, 607.913]


## Step 5 — All three kernels side by side

`link_contrast=False` so every kernel gets its own min/max (the SSB output is
~3 orders of magnitude smaller than ICOM).

In [47]:
quantem.widget.Show2D(
    [phases[k] for k in KERNELS],
    labels=[k.upper() for k in KERNELS],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
    link_contrast=False,
)

Show2D(3×512×512, idx=0, cmap=gray)

## What you just did

1. Loaded real 4D-STEM gold from Hugging Face → torch GPU + numpy `Dataset4dstem`.
2. Browsed with `Show4DSTEM`.
3. BF + DF in one `Show2D` widget (independent contrast) via `Dataset4dstem.get_virtual_image`.
4. DPC via upstream `CenterOfMassOriginModel.calculate_origin()`.
5. Found scan-detector rotation by minimizing CoM curl (Step 3b).
6. Built `DirectPtychography` with that rotation; ran a 30-trial optuna fit of C10/C12/phi12; swept ICOM / parallax / SSB.

| Method | What it uses |
|---|---|
| BF, DF | counts inside / outside the BF disk |
| DPC (CoM_row, CoM_col) | first moment per CBED |
| ICOM, parallax, SSB | full CBED at every scan position; aberration-corrected |

Takeaway: phase retrieval (especially SSB once aberrations are fit) recovers
contrast + resolution that BF/DF can not access at the same dose.

## Try next

- Swap to `gold_512_npy_bin4` for a 4× finer detector.
- Increase optuna trials past 30 if aberration loss isn't converging.
- v2 will add iterative ptycho (`PtychoLite`) for the highest-resolution phase.